# Instalação e Importação de Bibliotecas

In [0]:
%pip install databricks-feature-engineering
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, FunctionTransformer, MinMaxScaler
import mlflow
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier


# Amostragem

In [0]:
# Função para importar query SQL de um arquivo
def import_query(path):
    with open(path) as f:
        return f.read()

# Carrega query SQL
query = import_query("fl_inad.sql")
df = spark.sql(query)

# Lista de FeatureLookups para buscar features no Feature Store
feature_lookups = [
    FeatureLookup(table_name="feature_store.credit_score.fs_cadastral", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_temporal", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_historico_financeiro", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_renda", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_funcionarios", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_historico_pagamentos", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"])
]

# Cria objeto FeatureEngineeringClient
fe = FeatureEngineeringClient()

# Cria o training_set com features e label
training_set = fe.create_training_set(df=df, feature_lookups=feature_lookups, label="FL_INAD")

# Exibe o dataframe do training_set
training_set.load_df().display()


In [0]:
df_train  = training_set.load_df().toPandas()

In [0]:
# Ordena o dataframe pelo tempo de referência
df_sorted = df_train.sort_values('DATA_REF').reset_index(drop=True)

# Define tamanhos dos conjuntos de treino, validação e teste
n = len(df_sorted)
n_train = int(0.6 * n)
n_val   = int(0.2 * n)
n_test  = n - n_train - n_val

# Split temporal: primeiro 60% para treino, 20% para validação, 20% para teste
train_df = df_sorted.iloc[:n_train]
val_df   = df_sorted.iloc[n_train:n_train + n_val]
test_df  = df_sorted.iloc[n_train + n_val:]

# Separa features e target
X_train, y_train = train_df.drop(columns=['FL_INAD'], errors='ignore'), train_df['FL_INAD']
X_val,   y_val   = val_df.drop(columns=['FL_INAD'], errors='ignore'),   val_df['FL_INAD']
X_test,  y_test  = test_df.drop(columns=['FL_INAD'], errors='ignore'),  test_df['FL_INAD']

# Exploração

In [0]:
df_train_explore = X_train.copy()
df_train_explore["target"] = y_train

describe = (
    df_train_explore
    .groupby("target")
    .mean(numeric_only=True)
    .T
)

describe["variable"] = describe.index
describe["ratio"] = describe[1] / describe[0].replace(0, np.nan)

cols = ["variable"] + [col for col in describe.columns if col != "variable"]
describe = describe[cols]

display(describe)

In [0]:
# Calcula a porcentagem de valores NaN por coluna
nan_pct = X_train.isna().mean() * 100

# Filtra apenas colunas com NaN e ordena decrescentemente
nan_pct = nan_pct[nan_pct > 0].sort_values(ascending=False)

# Converte para DataFrame para visualização
nan_pct_df = nan_pct.to_frame('Porcentagem_NaN').reset_index().rename(columns={'index': 'Coluna'})

display(nan_pct_df)

In [0]:
# Imputador de valores nulos
class NullImputer(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        X = X.copy()
        X["PORTE"] = X["PORTE"].fillna("DESCONHECIDO")
        X["DDD"] = pd.to_numeric(X["DDD"], errors="coerce")

        # Modas para imputação de categorias
        self.estado_moda_ = X["ESTADO"].mode()[0]
        self.cep_moda_ = X["CEP_2_DIG"].mode()[0]
        self.regiao_moda_ = X["REGIAO"].mode()[0]
        self.ddd_moda_ = X["DDD"].mode()[0]
        self.ddd_estado_ = (
            X.groupby("ESTADO")["DDD"]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .to_dict()
        )

        # Imputação por média de porte e global para variáveis de nível
        self.cols_nivel = [
            'MED_RENDA_3M', 'MIN_RENDA_3M', 'MAX_RENDA_3M', 'SOMA_RENDA_3M',
            'MED_RENDA_6M', 'MIN_RENDA_6M', 'MAX_RENDA_6M', 'SOMA_RENDA_6M',
            'MED_RENDA_1A', 'MIN_RENDA_1A', 'MAX_RENDA_1A', 'SOMA_RENDA_1A',
            'MED_RENDA_VIDA', 'MIN_RENDA_VIDA', 'MAX_RENDA_VIDA',
        ]
        self.media_porte_ = {}
        self.media_global_ = {}
        for col in self.cols_nivel:
            self.media_porte_[col] = X.groupby("PORTE")[col].mean().to_dict()
            self.media_global_[col] = X[col].mean()

        # Constantes para imputação
        self.fill_zero_ = [
            'DIAS_ANTECIPADO_MEDIA_3M',
            'DIAS_ANTECIPADO_MEDIA_6M',
            'DIAS_ANTECIPADO_MEDIA_12M',
            'DIAS_ANTECIPADO_MEDIA_VIDA',
            'CRESCIMENTO_FUNC_3M',
            'CRESCIMENTO_FUNC_6M',
            'CRESCIMENTO_FUNC_12M',
            'CRESCIMENTO_FUNC_VIDA',
            'FLAG_PORTE_AUSENTE',
            'FLAG_HISTORICO_3M',
            'FLAG_HISTORICO_6M',
            'FLAG_HISTORICO_12M',
            'QT_SAFRAS_HISTORICO',
            'FLAG_HISTORICO_VARIACAO',
            'MAIOR_CRESCIMENTO_MENSAL',
            'MAIOR_QUEDA_MENSAL',
        ]
        self.fill_9999_ = [
            'DIAS_DESDE_ULTIMO_ATRASO',
            'DIAS_DESDE_ULTIMA_INADIMPLENCIA',
            'DIAS_ULT_PAG',
        ]
        self.fill_menos1_ = [
            'NO_FUNCIONARIOS_ATUAL',
            'NO_FUNCIONARIOS_3M',
            'NO_FUNCIONARIOS_6M',
            'NO_FUNCIONARIOS_12M',
            'NO_FUNCIONARIOS_VIDA',
        ]

        # Variáveis de variação e std
        self.cols_variacao = [
            'CRESCIMENTO_ABS_RENDA_3M', 'CRESCIMENTO_PERC_RENDA_3M',
            'CRESCIMENTO_ABS_RENDA_6M', 'CRESCIMENTO_PERC_RENDA_6M',
            'CRESCIMENTO_ABS_RENDA_1A', 'CRESCIMENTO_PERC_RENDA_1A',
            'MIN_DIFF_VALOR_RENDA_3M', 'MED_DIFF_VALOR_RENDA_3M', 'MAX_DIFF_VALOR_RENDA_3M',
            'MIN_RAZAO_VALOR_RENDA_3M', 'MED_RAZAO_VALOR_RENDA_3M', 'MAX_RAZAO_VALOR_RENDA_3M',
            'MIN_DIFF_VALOR_RENDA_6M', 'MED_DIFF_VALOR_RENDA_6M', 'MAX_DIFF_VALOR_RENDA_6M',
            'MIN_RAZAO_VALOR_RENDA_6M', 'MED_RAZAO_VALOR_RENDA_6M', 'MAX_RAZAO_VALOR_RENDA_6M',
            'MIN_DIFF_VALOR_RENDA_1A', 'MED_DIFF_VALOR_RENDA_1A', 'MAX_DIFF_VALOR_RENDA_1A',
            'MIN_RAZAO_VALOR_RENDA_1A', 'MED_RAZAO_VALOR_RENDA_1A', 'MAX_RAZAO_VALOR_RENDA_1A',
            'MIN_DIFF_VALOR_RENDA_VIDA', 'MED_DIFF_VALOR_RENDA_VIDA', 'MAX_DIFF_VALOR_RENDA_VIDA',
            'MIN_RAZAO_VALOR_RENDA_VIDA', 'MED_RAZAO_VALOR_RENDA_VIDA', 'MAX_RAZAO_VALOR_RENDA_VIDA',
        ]
        self.cols_std = [
            'STD_RENDA_3M', 'STD_RENDA_6M', 'STD_RENDA_1A', 'STD_RENDA_VIDA',
        ]

        # Cascatas para imputação sequencial
        self.cascatas = {
            'DIAS_ATRASO_MEDIA':  ['DIAS_ATRASO_MEDIA_3M', 'DIAS_ATRASO_MEDIA_6M', 'DIAS_ATRASO_MEDIA_12M'],
            'DIAS_ATRASO_MIN':    ['DIAS_ATRASO_MIN_3M', 'DIAS_ATRASO_MIN_6M', 'DIAS_ATRASO_MIN_12M'],
            'DIAS_ATRASO_MAX':    ['DIAS_ATRASO_MAX_3M', 'DIAS_ATRASO_MAX_6M', 'DIAS_ATRASO_MAX_12M'],
            'DIAS_EMISSAO_PAGAMENTO_MEDIA': [
                'DIAS_EMISSAO_PAGAMENTO_MEDIA_3M',
                'DIAS_EMISSAO_PAGAMENTO_MEDIA_6M',
                'DIAS_EMISSAO_PAGAMENTO_MEDIA_12M'
            ],
            'VALOR_COBRANCA_DIA_MEDIA': [
                'VALOR_COBRANCA_DIA_MEDIA_3M',
                'VALOR_COBRANCA_DIA_MEDIA_6M',
                'VALOR_COBRANCA_DIA_MEDIA_12M',
                'VALOR_COBRANCA_DIA_MEDIA_VIDA'
            ],
            'VALOR_A_PAGAR_MIN': [
                'VALOR_A_PAGAR_MIN_3M',
                'VALOR_A_PAGAR_MIN_6M',
                'VALOR_A_PAGAR_MIN_12M',
                'VALOR_A_PAGAR_MIN_VIDA'
            ],
            'VALOR_A_PAGAR_MEDIA': [
                'VALOR_A_PAGAR_MEDIA_3M',
                'VALOR_A_PAGAR_MEDIA_6M',
                'VALOR_A_PAGAR_MEDIA_12M',
                'VALOR_A_PAGAR_MEDIA_VIDA'
            ],
            'VALOR_A_PAGAR_MAX': [
                'VALOR_A_PAGAR_MAX_3M',
                'VALOR_A_PAGAR_MAX_6M',
                'VALOR_A_PAGAR_MAX_12M',
                'VALOR_A_PAGAR_MAX_VIDA'
            ],
            'QT_ATRASO_MIN': ['QT_ATRASO_MIN_3M', 'QT_ATRASO_MIN_6M', 'QT_ATRASO_MIN_12M'],
            'QT_ATRASO_MEDIA': ['QT_ATRASO_MEDIA_3M', 'QT_ATRASO_MEDIA_6M', 'QT_ATRASO_MEDIA_12M'],
            'QT_ATRASO_MAX': ['QT_ATRASO_MAX_3M', 'QT_ATRASO_MAX_6M', 'QT_ATRASO_MAX_12M'],
            'PCT_EM_DIA': ['PCT_EM_DIA_3M', 'PCT_EM_DIA_6M', 'PCT_EM_DIA_12M'],
            'PCT_FORA_INADIMPLENCIA': ['PCT_FORA_INADIMPLENCIA_3M', 'PCT_FORA_INADIMPLENCIA_6M', 'PCT_FORA_INADIMPLENCIA_12M'],
            'PRAZO_EMISSAO_VENCIMENTO_MEDIA': [
                'PRAZO_EMISSAO_VENCIMENTO_MEDIA_3M',
                'PRAZO_EMISSAO_VENCIMENTO_MEDIA_6M',
                'PRAZO_EMISSAO_VENCIMENTO_MEDIA_12M'
            ],
            'PRAZO_EMISSAO_VENCIMENTO_MAX': [
                'PRAZO_EMISSAO_VENCIMENTO_MAX_3M',
                'PRAZO_EMISSAO_VENCIMENTO_MAX_6M',
                'PRAZO_EMISSAO_VENCIMENTO_MAX_12M'
            ],
            'PRAZO_EMISSAO_VENCIMENTO_MIN': [
                'PRAZO_EMISSAO_VENCIMENTO_MIN_3M',
                'PRAZO_EMISSAO_VENCIMENTO_MIN_6M',
                'PRAZO_EMISSAO_VENCIMENTO_MIN_12M'
            ],
        }
        self.medianas_cascata_ = {}
        for _, cols in self.cascatas.items():
            serie = X[cols].bfill(axis=1).iloc[:,0]
            med = serie.median()
            if pd.isna(med):
                med = 0
            self.medianas_cascata_[cols[0]] = med

        # Medianas para colunas específicas
        cols_restantes = [
            'VALOR_COBRANCA_DIA_MEDIA_6M',
            'VALOR_A_PAGAR_MEDIA_VIDA',
            'VALOR_A_PAGAR_MIN_VIDA',
            'VALOR_A_PAGAR_MAX_VIDA',
            'VALOR_A_PAGAR_MAX_6M',
            'VALOR_A_PAGAR_MIN_6M',
            'VALOR_A_PAGAR_MEDIA_6M',
            'DIAS_EMISSAO_PAGAMENTO_MEDIA_6M',
            'PRAZO_EMISSAO_VENCIMENTO_MEDIA_6M',
            'PRAZO_EMISSAO_VENCIMENTO_MIN_6M',
            'PCT_EM_DIA_6M',
            'PRAZO_EMISSAO_VENCIMENTO_MAX_6M',
            'PCT_FORA_INADIMPLENCIA_6M',
            'DIAS_ATRASO_MAX_6M',
            'DIAS_ATRASO_MIN_6M',
            'DIAS_ATRASO_MEDIA_6M',
            'QT_ATRASO_MAX_6M',
            'QT_ATRASO_MIN_6M',
            'QT_ATRASO_MEDIA_6M',
            'VALOR_COBRANCA_DIA_MEDIA_12M',
            'PRAZO_EMISSAO_VENCIMENTO_MAX_12M',
            'PRAZO_EMISSAO_VENCIMENTO_MIN_12M',
            'DIAS_EMISSAO_PAGAMENTO_MEDIA_12M',
            'PRAZO_EMISSAO_VENCIMENTO_MEDIA_12M',
            'VALOR_A_PAGAR_MEDIA_12M',
            'PCT_FORA_INADIMPLENCIA_12M',
            'PCT_EM_DIA_12M',
            'VALOR_A_PAGAR_MAX_12M',
            'VALOR_A_PAGAR_MIN_12M',
            'DIAS_ATRASO_MAX_12M',
            'DIAS_ATRASO_MIN_12M',
            'DIAS_ATRASO_MEDIA_12M',
            'QT_ATRASO_MAX_12M',
            'QT_ATRASO_MIN_12M',
            'QT_ATRASO_MEDIA_12M',
            'VALOR_COBRANCA_DIA_MEDIA_VIDA'
        ]
        self.cols_mediana = cols_restantes + [
            'RAZAO_RENDA_POR_FUNCIONARIO',
            'RAZAO_TAXA_VALOR_A_PAGAR',
            'DIF_PARA_MEDIA_PORTE'
        ]
        self.medianas_ = {}
        for col in self.cols_mediana:
            if col in X.columns:
                med = X[col].median()
                if pd.isna(med):
                    med = 0
                self.medianas_[col] = med

        return self

    def transform(self, X):
        X = X.copy()
        X["DDD"] = pd.to_numeric(X["DDD"], errors="coerce")
        X["PORTE"] = X["PORTE"].fillna("DESCONHECIDO")
        X["SEGMENTO_INDUSTRIAL"] = X["SEGMENTO_INDUSTRIAL"].fillna("desconhecido")
        X["DOMINIO_EMAIL"] = X["DOMINIO_EMAIL"].fillna("DESCONHECIDO")
        X["DDD"] = X["DDD"].mask(
            X["DDD"].isna(),
            X["ESTADO"].map(self.ddd_estado_)
        )
        X["DDD"] = X["DDD"].fillna(self.ddd_moda_)
        X["ESTADO"] = X["ESTADO"].fillna(self.estado_moda_)
        X["CEP_2_DIG"] = X["CEP_2_DIG"].fillna(self.cep_moda_)
        X["REGIAO"] = X["REGIAO"].fillna(self.regiao_moda_)

        # Corrige FLAG_SEM_HISTORICO_FUNCIONARIOS
        X["FLAG_SEM_HISTORICO_FUNCIONARIOS"] = (
            X["FLAG_PORTE_AUSENTE"].isna().astype("int8")
        )

        # Imputação por constantes
        for col in self.fill_zero_:
            if col in X.columns:
                X[col] = X[col].fillna(0)
        for col in self.fill_9999_:
            if col in X.columns:
                X[col] = X[col].fillna(9999)
        for col in self.fill_menos1_:
            if col in X.columns:
                X[col] = X[col].fillna(-1)

        # Imputação por média de porte e global
        for col in self.cols_nivel:
            X[f"FLAG_{col}_IMPUTADA"] = X[col].isna().astype("int8")
            medias = X["PORTE"].map(self.media_porte_[col])
            X[col] = (
                X[col]
                .fillna(medias)
                .fillna(self.media_global_[col])
            )

        # Flags e imputação para variáveis de variação e std
        for col in self.cols_variacao:
            if col in X.columns:
                X[f"FLAG_{col}_AUSENTE"] = X[col].isna().astype("int8")
                X[col] = X[col].fillna(0)
        for col in self.cols_std:
            if col in X.columns:
                X[f"FLAG_{col}_AUSENTE"] = X[col].isna().astype("int8")
                X[col] = X[col].fillna(0)

        # Imputação para MESES_CONSECUTIVOS_QUEDA
        if "MESES_CONSECUTIVOS_QUEDA" in X.columns:
            X["FLAG_SEM_HISTORICO_RENDA"] = (
                X["MESES_CONSECUTIVOS_QUEDA"]
                .isna()
                .astype("int8")
            )
            X["MESES_CONSECUTIVOS_QUEDA"] = (
                X["MESES_CONSECUTIVOS_QUEDA"]
                .fillna(0)
            )

        # Imputação por cascata
        for _, cols in self.cascatas.items():
            principal = cols[0]
            X[f"FLAG_{principal}_IMPUTADA_CASCATA"] = (
                X[principal].isna().astype("int8")
            )
            resultado = X[cols].bfill(axis=1).iloc[:,0]
            X[principal] = resultado.fillna(
                self.medianas_cascata_[principal]
            )

        # Imputação por mediana
        for col, med in self.medianas_.items():
            if col in X.columns:
                X[col] = X[col].fillna(med)
        return X

# Extração de features de data
def add_date_features(X):
    X = X.copy()
    X['ANO_REF'] = pd.to_datetime(X['DATA_REF']).dt.year
    X['MES_REF'] = pd.to_datetime(X['DATA_REF']).dt.month
    X['ANO_CADASTRO'] = pd.to_datetime(X['DATA_CADASTRO']).dt.year
    X['MES_CADASTRO'] = pd.to_datetime(X['DATA_CADASTRO']).dt.month
    X = X.drop(columns=['DATA_REF', 'DATA_CADASTRO'])
    return X

# Conversão ordinal para PORTE
def add_porte_ordinal(X):
    X = X.copy()
    porte_ordem = ['PEQUENO', 'MEDIO', 'GRANDE', 'DESCONHECIDO']
    X['PORTE'] = X['PORTE'].map({v: i for i, v in enumerate(porte_ordem)})
    return X

# Label encoding customizado para ESTADO e DDD
class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.estado_map_ = None
        self.ddd_map_ = None

    def fit(self, X, y=None):
        X = X.copy()
        self.estado_map_ = {
            v: i
            for i, v in enumerate(sorted(X["ESTADO"].astype(str).unique()))
        }
        self.ddd_map_ = {
            v: i
            for i, v in enumerate(sorted(X["DDD"].astype(str).unique()))
        }
        return self

    def transform(self, X):
        X = X.copy()
        X["ESTADO_LABEL"] = (
            X["ESTADO"]
            .astype(str)
            .map(self.estado_map_)
            .fillna(-1)
            .astype(int)
        )
        X["DDD_LABEL"] = (
            X["DDD"]
            .astype(str)
            .map(self.ddd_map_)
            .fillna(-1)
            .astype(int)
        )
        X = X.drop(columns=["ESTADO", "DDD"])
        return X

# Pré-processamento para definição de colunas numéricas
tmp = NullImputer().fit_transform(X_train)
tmp = add_date_features(tmp)
tmp = add_porte_ordinal(tmp)
tmp = LabelEncoderTransformer().fit_transform(tmp)
num_cols = tmp.select_dtypes(include=np.number).columns.tolist()
del tmp


# Modificação

In [0]:
# Pipeline de pré-processamento
preprocess_pipeline = Pipeline([
    ("null_imputer", NullImputer()),
    ("date_features", FunctionTransformer(add_date_features, validate=False)),
    ("porte_ordinal", FunctionTransformer(add_porte_ordinal, validate=False)),
    ("label_encodings", LabelEncoderTransformer()),
    ("column_transform", ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["SEGMENTO_INDUSTRIAL", "REGIAO", "DOMINIO_EMAIL"]),
            ("num", MinMaxScaler(), num_cols)
        ],
        remainder="passthrough"
    ))
])

# Modelagem

In [0]:
# Define experimento MLflow
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo XGBoost
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("xgb_model", XGBClassifier(use_label_encoder=False, eval_metric="logloss"))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calcula AUC para cada conjunto
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)

In [0]:
# Define experimento MLflow
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo Logistic Regression
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("logreg_model", LogisticRegression(max_iter=1000))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calcula AUC para cada conjunto
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)

In [0]:
# Define experimento MLflow
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo Random Forest
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("rf_model", RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calcula AUC para cada conjunto
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)

# Modelo Final

In [0]:
# Inicia run MLflow
with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo XGBoost
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("xgb_model", XGBClassifier(
            use_label_encoder=False,
            eval_metric="logloss" 
        ))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", roc_auc_score(y_train, y_train_pred))
    mlflow.log_metric("auc_val", roc_auc_score(y_val, y_val_pred))
    mlflow.log_metric("auc_test", roc_auc_score(y_test, y_test_pred))

    # Salva modelo no MLflow
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="model"
    )

    print(run.info.run_id)  # Exibe run_id do MLflow

##